# 2C · 首例 FreeSurfer 结构体积提取

病例：**sub-mindo304_ses-placebo**。这是独立于 2B / 编码器的结构测量流程。

**逐格运行，不要 Run all。** 本文件最初发布时未执行；没有真实 FreeSurfer 结果。
输入是 native intake_v2 T1，不是去颅骨/模板空间/96³ 图。
候选版本为 FreeSurfer 8.2.0，`FS_V8_XOPTS=0` 传统流程；全批次不能静默换版本或模式。

运行前：有效 RAM ≥30 GiB（约32 GB机器）、本地空闲磁盘≥40 GiB、正确项目 Drive、个人许可。
这是一道保守资源门槛，达标不保证成功。GPU不能代替系统RAM。
本 notebook 不替你接受许可，安装和三段重建均有默认关闭的开关。
不要运行旧 notebook 的全部历史安装/启动格。

预计交付：68皮层+14皮层下原始mm³体积、独立eTIV、原始stats、QC与运行记录。
只做预处理，不训练分类器、不计算AUC，不把技术成功记成人工QC通过。


## P1 · 获取固定代码快照（只下载小型公开代码）

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, urllib.request, subprocess, importlib.util
COMMIT = 'd8bdfa7a125c9153943081f436eaf13966fea65d'
EXPECTED = {'configs/freesurfer_pilot.json': 'cf4bae2f9360d49f08ce5b2d10041031ca45a2474221d0cbe2a84a759cab842f', 'schemas/t1_volume_names.json': '5a2fffd7404afa7b7e8e6302d6c76593985317fd49d60128d0b9c747fc4b0c99', 'scripts/freesurfer_pilot.py': '7aab31025ac41c61dff99c182c1b8a4eed066884768e82998c789874ca8215fe', 'scripts/freesurfer_volumes.py': 'fae7c75809aeaaa5f0df8ff8d497c3cf7454bd2e48b07c59e74daff2194f0b95', 'scripts/freesurfer_qc.py': '1b08d88d01cdbb44f4eca980cd7d38c063714fa29a97cdd70ad659d8e9db0ca8'}
CODE_ROOT = Path('/content/FYP_MRI_2C_Code_' + COMMIT[:12])
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for relative, expected in EXPECTED.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        content = target.read_bytes()
    else:
        url = 'https://raw.githubusercontent.com/bykarl/MRI-classification/' + COMMIT + '/' + relative
        with urllib.request.urlopen(url, timeout=120) as response:
            content = response.read()
        assert hashlib.sha256(content).hexdigest() == expected, relative + ': download SHA mismatch'
        with target.open('xb') as stream:
            stream.write(content)
    assert hashlib.sha256(content).hexdigest() == expected, relative + ': existing code differs; do not overwrite'
snapshot = {'repository': 'bykarl/MRI-classification', 'commit': COMMIT, 'files_sha256': EXPECTED}
snapshot_file = CODE_ROOT / 'source_snapshot.json'
if snapshot_file.exists():
    assert json.loads(snapshot_file.read_text()) == snapshot
else:
    snapshot_file.write_text(json.dumps(snapshot, indent=2))
sys.path.insert(0, str(CODE_ROOT / 'scripts'))
import freesurfer_pilot as fs
CONFIG = CODE_ROOT / 'configs/freesurfer_pilot.json'
cfg = fs.load_config(CONFIG)
def call_script(name, *args, python=None):
    subprocess.run([str(python or sys.executable), str(CODE_ROOT / 'scripts' / name),
                    '--config', str(CONFIG), *map(str, args)], check=True)
print('Pinned source verified:', COMMIT)
print('No software installation or reconstruction started.')


## P2 · 检查当前资源（不安装、不处理影像）
如果 BLOCKED，不能启动安装和重建；可继续 P3 核对已备份输入。

In [ ]:
resource = fs.runtime_info()
print(json.dumps(resource, indent=2))
RESOURCE_OK = (resource['os_id'] == 'ubuntu' and resource['os_version'] in cfg['packages']
               and resource['architecture'] in ('x86_64', 'amd64') and resource['avx_present']
               and resource['effective_memory_gib'] >= cfg['minimum_memory_gib']
               and resource['free_disk_gib'] >= cfg['minimum_free_disk_gib'])
print('Resource gate:', 'PASS' if RESOURCE_OK else 'BLOCKED: review OS/CPU/RAM/disk before installation')


## P3 · 挂载项目 Drive，逐文件核对 SHA
授权由本人完成。只核对既有 raw T1 与 native intake，不重新下载全队列。

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
input_checks = {}
for relative_key, hash_key in [('raw_relative', 'raw_sha256'), ('input_relative', 'input_sha256')]:
    path = Path(cfg['persistent_root']) / cfg[relative_key]
    assert path.is_file(), 'Project Drive input missing: ' + str(path)
    actual = fs.digest(path)
    assert actual == cfg[hash_key], 'Input SHA mismatch: ' + str(path)
    input_checks[relative_key] = {'relative_path': cfg[relative_key], 'sha256': actual, 'status': 'pass'}
print(json.dumps(input_checks, indent=2))
preparation_report = {'checked_at_utc': fs.now(), 'source_commit': COMMIT,
                      'resource': fs.runtime_info(), 'resource_gate_pass': RESOURCE_OK,
                      'inputs': input_checks, 'license_present': Path(cfg['fs_license']).is_file(),
                      'freesurfer_installed_tested': False, 'reconstruction_started': False,
                      'qc_status': 'pending', 'training_ready': False}
report_name = 'preflight_' + preparation_report['checked_at_utc'].replace(':', '').replace('.', '_') + '.json'
report_path = CODE_ROOT / report_name
fs.write_new(report_path, preparation_report)
fs.copy_checked(report_path, Path(cfg['persistent_root']) / 'freesurfer_pilot/preparation' / report_name)
print('Input verification and preparation report saved with Drive SHA readback.')


## P4 · 检查个人许可

通过Colab左侧 Files > Upload to session storage上传已有个人license.txt，再运行下一格检查。内容不显示、不归档、不进入Git。


In [ ]:
license_path=Path(cfg['fs_license'])
assert license_path.is_file() and license_path.stat().st_size>0, 'Upload your existing personal license.txt through Colab Files > Upload to session storage, then rerun this cell.'
license_path.chmod(0o600)
print('Private license present. Contents are not displayed, archived or sent to Git.')


## P5 · 安装官方 FreeSurfer（默认关闭）
资源及输入检查通过、本人已阅读并接受许可后，才将下面开关改成 True。安装包约8–9GB，需等待下载和系统依赖安装。失败先看报错，不启动重建。

In [ ]:
I_HAVE_READ_AND_ACCEPTED_FREESURFER_LICENSE = False
if not I_HAVE_READ_AND_ACCEPTED_FREESURFER_LICENSE:
    raise RuntimeError('Not installed. Personally review/accept the official license before enabling this cell.')
fs.require_ready(fs.preflight(cfg, require_license=True))
call_script('freesurfer_pilot.py', 'install', '--license-accepted')
print('Vendor install/version/shared-library checks completed; no reconstruction started.')


## P6 · 准备隔离QC环境

QC使用独立Python3.10及固定numpy/nibabel/matplotlib。对子进程显式设置Agg后端，避免继承Colab的inline后端；不修改内核的科学计算包。


In [ ]:
QC_ENV=Path('/content/FYP_MRI_2C_QC_py310')
QC_PROCESS_ENV={**os.environ,'MPLBACKEND':'Agg'}
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','uv'],check=True)
uv=[sys.executable,'-m','uv']
if not QC_ENV.exists():
    subprocess.run([*uv,'venv','--python','3.10',str(QC_ENV)],check=True)
QC_PYTHON=QC_ENV/'bin/python'
assert QC_PYTHON.is_file(),'QC interpreter missing'
for command in ([*uv,'pip','install','--python',str(QC_PYTHON),'numpy==1.26.4','nibabel==5.3.2','matplotlib==3.9.4'],
                [str(QC_PYTHON),'-c','import numpy,nibabel,matplotlib; print(numpy.__version__,nibabel.__version__,matplotlib.__version__); assert matplotlib.get_backend().lower()=="agg"']):
    result=subprocess.run(command,env=QC_PROCESS_ENV,capture_output=True,text=True)
    print(result.stdout); print(result.stderr)
    result.check_returncode()
freeze=subprocess.run([*uv,'pip','freeze','--python',str(QC_PYTHON)],capture_output=True,text=True,check=True)
freeze_path=CODE_ROOT/('qc_environment_'+fs.now().replace(':','').replace('.','_')+'.txt')
freeze_path.write_text(freeze.stdout)
fs.copy_checked(freeze_path,Path(cfg['persistent_root'])/'freesurfer_pilot/preparation'/freeze_path.name)
print('PASS: QC imports/Agg backend and Drive freeze readback. No reconstruction started.')


## C0 · 初始化新的首例运行（默认关闭）

确认软件配方后启用。记下输出的 RUN_ID；**不要重复运行初始化来继续已有任务**。
若从检查点恢复，跳过本格，使用最后的 R 格。


In [ ]:
INITIALIZE_NEW_RUN = False
if not INITIALIZE_NEW_RUN:
    raise RuntimeError('Initialization is disabled. Enable explicitly when prepared to create a NEW run.')
RUN_ID = 'fs820_' + fs.now().split('.')[0].replace(':', '').replace('-', '').replace('+', '_')
call_script('freesurfer_pilot.py', 'init', '--run-id', RUN_ID)
print('SAVE THIS RUN_ID:', RUN_ID)


## C1 · 内部校正、配准与去颅骨
单独启用本格；等待正常结束和 Drive 检查点读回通过后再继续。只记录日志和阶段结果，不自动批准 QC。

In [ ]:
START_AUTORECON1 = False
if not START_AUTORECON1:
    raise RuntimeError('This reconstruction stage is disabled. Start it explicitly when ready.')
assert 'RUN_ID' in globals(), 'Set the existing RUN_ID or initialize/restore once first'
call_script('freesurfer_pilot.py', 'run-stage', '--run-id', RUN_ID, '--stage', 'autorecon1')
call_script('freesurfer_pilot.py', 'status', '--run-id', RUN_ID)


## C2 · 分割与主要表面重建
单独启用本格；等待正常结束和 Drive 检查点读回通过后再继续。只记录日志和阶段结果，不自动批准 QC。

In [ ]:
START_AUTORECON2 = False
if not START_AUTORECON2:
    raise RuntimeError('This reconstruction stage is disabled. Start it explicitly when ready.')
assert 'RUN_ID' in globals(), 'Set the existing RUN_ID or initialize/restore once first'
call_script('freesurfer_pilot.py', 'run-stage', '--run-id', RUN_ID, '--stage', 'autorecon2')
call_script('freesurfer_pilot.py', 'status', '--run-id', RUN_ID)


## C3 · 皮层分区与统计
单独启用本格；等待正常结束和 Drive 检查点读回通过后再继续。只记录日志和阶段结果，不自动批准 QC。

In [ ]:
START_AUTORECON3 = False
if not START_AUTORECON3:
    raise RuntimeError('This reconstruction stage is disabled. Start it explicitly when ready.')
assert 'RUN_ID' in globals(), 'Set the existing RUN_ID or initialize/restore once first'
call_script('freesurfer_pilot.py', 'run-stage', '--run-id', RUN_ID, '--stage', 'autorecon3')
call_script('freesurfer_pilot.py', 'status', '--run-id', RUN_ID)


## Q · 生成真实分割叠图
C3 成功后运行。抽样PNG不等于完整表面/全体积人工审核；标签来自算法，不是人工GT。

In [ ]:
assert 'RUN_ID' in globals() and 'QC_PYTHON' in globals()
call_script('freesurfer_qc.py', '--run-id', RUN_ID, python=QC_PYTHON)
from IPython.display import display, Image
display(Image(filename=str(fs.run_dir(cfg, RUN_ID) / 'qc/freesurfer_regions.png')))


## E · 导出待审核结构特征
保存原始 stats、82项mm³体积和独立eTIV。维持pending，不在这里做ICV校正、缩放或分类。重复导出不会覆盖已有draft。

In [ ]:
call_script('freesurfer_volumes.py', 'export', '--run-id', RUN_ID)
print('Draft:', fs.run_dir(cfg, RUN_ID) / 'structure/draft')
print('Persistent copy:', fs.persistent_run(cfg, RUN_ID) / 'structure/draft')


## A · 人工审核后正式交接（默认关闭）

由具名审核者真实检查完整体积、左右white/pial表面与Talairach配准。
在 draft 的 `qc_review.template.json` 基础上另存审核文件，不自动填True。
与队友确认本配方和全项目通用 `pipeline_id`，不要只为FreeSurfer随意造一个ID。
结构阶段ready仍不代表整个病例training_ready。


In [ ]:
FINALIZE_AFTER_HUMAN_QC = False
REVIEW_FILE = ''   # actual reviewed JSON file, never an unreviewed template
AGREED_PIPELINE_ID = ''   # team-wide ID shared with T1/fMRI handoff
TEAM_CONFIRMED_RECIPE = False
if not (FINALIZE_AFTER_HUMAN_QC and TEAM_CONFIRMED_RECIPE and REVIEW_FILE and AGREED_PIPELINE_ID):
    raise RuntimeError('Human QC, an actual review file and the agreed pipeline/recipe are required.')
call_script('freesurfer_volumes.py', 'finalize', '--run-id', RUN_ID,
            '--review', REVIEW_FILE, '--pipeline-id', AGREED_PIPELINE_ID, '--recipe-confirmed')


## S · 查看状态（不启动计算）

若会话刚重置，先运行P1，挂载Drive；本地运行目录需要从R恢复。
同一运行时内可把下面RUN_ID改成之前保存的值，仅检查状态；失败/中断先看log，不能盲删锁。


In [ ]:
assert 'RUN_ID' in globals(), 'Enter the saved RUN_ID before checking an existing run'
call_script('freesurfer_pilot.py', 'status', '--run-id', RUN_ID)


## R · 恢复已完成阶段（默认关闭）

只在确需恢复时使用，普通从头运行不运行本格。
新运行时先完成P1–P5，要求相同FreeSurfer build/OS/config。
填写 source run和最后成功阶段；恢复到新的run ID，之后只点下一个未完成阶段。
中断阶段尚未完成的进度不保证保留；不删除旧运行或锁，不并行重复运行同一病例。


In [ ]:
RESTORE_COMPLETED_CHECKPOINT = False
SOURCE_RUN_ID = ''
LAST_COMPLETED_STAGE = 'autorecon1'  # or autorecon2/autorecon3, actual completed marker required
NEW_RUN_ID = ''
if not (RESTORE_COMPLETED_CHECKPOINT and SOURCE_RUN_ID and NEW_RUN_ID):
    raise RuntimeError('Restore is disabled; supply the actual completed source and a NEW run ID.')
call_script('freesurfer_pilot.py', 'restore', '--run-id', NEW_RUN_ID,
            '--from-run', SOURCE_RUN_ID, '--stage', LAST_COMPLETED_STAGE)
RUN_ID = NEW_RUN_ID
call_script('freesurfer_pilot.py', 'status', '--run-id', RUN_ID)


### 阅读与边界

完整说明：仓库 `docs/STEP2C_RUNBOOK.md`。
仅已完成阶段有完整归档恢复；Colab自身会话限制仍可能中断长任务。
个人许可、原始MRI和分割二进制不进入Git；运行与产物保存在指定项目Drive。
软件安装、真实重建、耗时和质量需逐项实测，不能把准备代码当成已完成结果。
